In [1]:
import pandas as pd
import numpy as np

In [3]:
# mount to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
from datasets import load_dataset
project_root = '/content/drive/MyDrive/Intelligent-Agents-Project/code'
data_root = project_root + '/data/drugs.csv'
data = pd.read_csv(data_root)

In [15]:
data.head()

,Unnamed: 0,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level,Dataset,Category,question,answer
0,0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate,A,Interaction involving alimentary tract and met...,How do Naltrexone and Abacavir interact?,Naltrexone and Abacavir are reported to have a...
1,1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate,A,Interaction involving alimentary tract and met...,How do Abacavir and Orlistat interact?,Abacavir and Orlistat are reported to have a m...
2,2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Aluminum hydroxide and Dolutegravir int...,Aluminum hydroxide and Dolutegravir are report...
3,3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor,A,Interaction involving alimentary tract and met...,How do Aprepitant and Dolutegravir interact?,Aprepitant and Dolutegravir are reported to ha...
4,4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Attapulgite and Dolutegravir interact?,Attapulgite and Dolutegravir are reported to h...


In [16]:
data['generated_answer'] = ''
data.head()


,Unnamed: 0,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level,Dataset,Category,question,answer,generated_answer
0,0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate,A,Interaction involving alimentary tract and met...,How do Naltrexone and Abacavir interact?,Naltrexone and Abacavir are reported to have a...,
1,1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate,A,Interaction involving alimentary tract and met...,How do Abacavir and Orlistat interact?,Abacavir and Orlistat are reported to have a m...,
2,2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Aluminum hydroxide and Dolutegravir int...,Aluminum hydroxide and Dolutegravir are report...,
3,3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor,A,Interaction involving alimentary tract and met...,How do Aprepitant and Dolutegravir interact?,Aprepitant and Dolutegravir are reported to ha...,
4,4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Attapulgite and Dolutegravir interact?,Attapulgite and Dolutegravir are reported to h...,


In [17]:
# first check if the llm output gets just the level correct, by extracting it

In [18]:
data.groupby(['Level']).count()

,Unnamed: 0,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Dataset,Category,question,answer,generated_answer
Level,,,,,,,,,,
Major,33896,33896,33896,33896,33896,33896,33896,33896,33896,33896
Minor,10938,10938,10938,10938,10938,10938,10938,10938,10938,10938
Moderate,130367,130367,130367,130367,130367,130367,130367,130367,130367,130367
Unknown,47182,47182,47182,47182,47182,47182,47182,47182,47182,47182


In [19]:
import re

def extract_level(text):
    match = re.search(r"\b(major|moderate|minor|unknown)\b", text.lower())
    if match:
        return match.group(1).capitalize()
    return None

# call extract_level on the output column and check if the value matches the level column
data['Model Level Match']  = data['generated_answer'].apply(extract_level) == data['Level']
data

,Unnamed: 0,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level,Dataset,Category,question,answer,generated_answer,Model Level Match
0,0,DDInter1263,Naltrexone,DDInter1,Abacavir,Moderate,A,Interaction involving alimentary tract and met...,How do Naltrexone and Abacavir interact?,Naltrexone and Abacavir are reported to have a...,,False
1,1,DDInter1,Abacavir,DDInter1348,Orlistat,Moderate,A,Interaction involving alimentary tract and met...,How do Abacavir and Orlistat interact?,Abacavir and Orlistat are reported to have a m...,,False
2,2,DDInter58,Aluminum hydroxide,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Aluminum hydroxide and Dolutegravir int...,Aluminum hydroxide and Dolutegravir are report...,,False
3,3,DDInter112,Aprepitant,DDInter582,Dolutegravir,Minor,A,Interaction involving alimentary tract and met...,How do Aprepitant and Dolutegravir interact?,Aprepitant and Dolutegravir are reported to ha...,,False
4,4,DDInter138,Attapulgite,DDInter582,Dolutegravir,Major,A,Interaction involving alimentary tract and met...,How do Attapulgite and Dolutegravir interact?,Attapulgite and Dolutegravir are reported to h...,,False
...,...,...,...,...,...,...,...,...,...,...,...,...
222378,222378,DDInter667,Erlotinib,DDInter766,Fluticasone,Unknown,L,Interaction involving antineoplastic and immun...,How do Erlotinib and Fluticasone interact?,Erlotinib and Fluticasone are reported to have...,,False
222379,222379,DDInter789,Fulvestrant,DDInter766,Fluticasone,Unknown,L,Interaction involving antineoplastic and immun...,How do Fulvestrant and Fluticasone interact?,Fulvestrant and Fluticasone are reported to ha...,,False
222380,222380,DDInter1156,Mercaptopurine,DDInter766,Fluticasone,Unknown,L,Interaction involving antineoplastic and immun...,How do Mercaptopurine and Fluticasone interact?,Mercaptopurine and Fluticasone are reported to...,,False
222381,222381,DDInter810,Gefitinib,DDInter888,Hydrocortisone butyrate,Unknown,L,Interaction involving antineoplastic and immun...,How do Gefitinib and Hydrocortisone butyrate i...,Gefitinib and Hydrocortisone butyrate are repo...,,False


In [20]:
# use a transformer for sentence similarity
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_semantic_similarity(df, true_col="answer", gen_col="generated_answer"):
    # Convert to list
    true_texts = df[true_col].tolist()
    gen_texts = df[gen_col].tolist()

    # Encode all at once (much faster)
    true_embeddings = model.encode(true_texts, convert_to_tensor=True, show_progress_bar=True)
    gen_embeddings = model.encode(gen_texts, convert_to_tensor=True, show_progress_bar=True)

    # Compute cosine similarity row-wise
    cosine_scores = util.cos_sim(true_embeddings, gen_embeddings)

    # Extract diagonal (each true[i] vs gen[i])
    similarities = cosine_scores.diagonal().cpu().numpy()

    df["semantic_similarity"] = similarities

    return df



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
eval_df = compute_semantic_similarity(data)

print("Average Semantic Similarity:", eval_df["semantic_similarity"].mean())

eval_df.sort_values("semantic_similarity").head(10)[
    ["Drug_A", "Drug_B", "true_answer", "generated_answer", "semantic_similarity"]
]

Batches:   0%|          | 0/6950 [00:00<?, ?it/s]